# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bytebymanas/fly-rank-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip -q install duckdb

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("Connected to FlyRank warehouse ✅")

Connected to FlyRank warehouse ✅


In [2]:
REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

schema = con.sql(f"DESCRIBE SELECT * FROM {REL}").df()
schema[["column_name", "column_type"]]

,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


In [3]:
print("\n".join(schema["column_name"].tolist()))

report_date
client_hash_id
content_hash_id
client_has_gsc
client_has_ga4
gsc_data_available
ga4_data_available
gsc_impressions
gsc_clicks
gsc_sum_position
gsc_avg_position
ga4_pageviews
ga4_sessions
ga4_users
ga4_engaged_sessions
ga4_total_engagement_sec
sessions_organic
sessions_direct
sessions_referral
sessions_social
sessions_paid
sessions_ai
ai_chatgpt
ai_perplexity
ai_gemini
ai_copilot
ai_claude
ai_meta
ai_other
scroll_events
month


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents the daily performance of one content item for one client.

Time window: I am using the March 2026 partition, from 2026-03-01 to 2026-03-31.

In [5]:
q = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicate_count
FROM {REL}
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
"""

con.sql(q).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,duplicate_count


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Table
I am using `fact_content_daily_performance`, specifically the March 2026 partition.

### Feature
The five features I will use are `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `sessions_organic`, and `scroll_events`.

### Label / proxy
I will predict whether a content item's Google Search clicks decrease in the following 30-day outcome window. The label is `decline_label`.

### Context
`report_date`, `client_hash_id`, `content_hash_id`, `month`, `gsc_data_available`, and `ga4_data_available` are context fields used for filtering, grouping, joining, or checking availability.

### Excluded
I deliberately exclude future outcome metrics from the features because they are not known at the decision moment. I also exclude the client and content IDs because they are identifiers, not predictive features.

In [15]:
contract_fields = {
    "features": [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "sessions_organic",
        "scroll_events"
    ],
    "label_or_proxy": [
        "decline_label"
    ],
    "context": [
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "month",
        "gsc_data_available",
        "ga4_data_available"
    ],
    "excluded": [
        "future outcome metrics",
        "client_hash_id as a model feature",
        "content_hash_id as a model feature"
    ]
}

for bucket, fields in contract_fields.items():
    print(f"\n{bucket.upper()}:")
    for field in fields:
        print(f" - {field}")


FEATURES:
 - gsc_impressions
 - gsc_clicks
 - gsc_avg_position
 - sessions_organic
 - scroll_events

LABEL_OR_PROXY:
 - decline_label

CONTEXT:
 - report_date
 - client_hash_id
 - content_hash_id
 - month
 - gsc_data_available
 - ga4_data_available

EXCLUDED:
 - future outcome metrics
 - client_hash_id as a model feature
 - content_hash_id as a model feature


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [9]:
march_rel = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

q = f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions_30d,
    SUM(gsc_clicks) AS clicks_30d,
    AVG(NULLIF(gsc_avg_position, 0)) AS avg_position_30d,
    SUM(sessions_organic) AS organic_sessions_30d,
    SUM(scroll_events) AS scroll_events_30d
FROM {march_rel}
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
"""

features = con.sql(q).df()

print("Feature frame shape:", features.shape)
features.head()

Feature frame shape: (176738, 7)


,client_hash_id,content_hash_id,impressions_30d,clicks_30d,avg_position_30d,organic_sessions_30d,scroll_events_30d
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,77.0,0.0,4.888929,NaN,NaN
1,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,10849.0,22.0,8.240351,NaN,NaN
2,client_62f4a7e64f5e0096,content_e689bc511192751a,61.0,0.0,7.061594,NaN,NaN
3,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,705.0,1.0,6.155424,NaN,NaN
4,client_62f4a7e64f5e0096,content_40b10da45f4c1cb5,50.0,0.0,14.343567,NaN,NaN


In [10]:
april_rel = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet')"

q = f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_clicks) AS future_clicks_30d
FROM {april_rel}
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
"""

future = con.sql(q).df()

print("Future outcome rows:", future.shape)
future.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Future outcome rows: (194760, 3)


,client_hash_id,content_hash_id,future_clicks_30d
0,client_62f4a7e64f5e0096,content_143987cfdeaba4c0,0.0
1,client_62f4a7e64f5e0096,content_13a8105125458098,0.0
2,client_62f4a7e64f5e0096,content_e2bd76be7eed690d,0.0
3,client_62f4a7e64f5e0096,content_86ab16840c4e0e1a,1.0
4,client_62f4a7e64f5e0096,content_3f87f49c36774e23,0.0


In [11]:
# Merge March features with April outcome
model_df = features.merge(
    future,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Future outcome: 1 = clicks declined in April, 0 = did not decline
model_df["decline_label"] = (
    model_df["future_clicks_30d"] < model_df["clicks_30d"]
).astype(int)

feature_cols = [
    "impressions_30d",
    "clicks_30d",
    "avg_position_30d",
    "organic_sessions_30d",
    "scroll_events_30d"
]

final_features = model_df[feature_cols + ["decline_label"]].copy()

print("Final feature frame shape:", final_features.shape)
print("Label distribution:")
print(final_features["decline_label"].value_counts())
final_features.head()

Final feature frame shape: (158549, 6)
Label distribution:
decline_label
0    114244
1     44305
Name: count, dtype: int64


,impressions_30d,clicks_30d,avg_position_30d,organic_sessions_30d,scroll_events_30d,decline_label
0,77.0,0.0,4.888929,NaN,NaN,0
1,10849.0,22.0,8.240351,NaN,NaN,0
2,61.0,0.0,7.061594,NaN,NaN,0
3,705.0,1.0,6.155424,NaN,NaN,1
4,50.0,0.0,14.343567,NaN,NaN,0


### Five features

1. `impressions_30d` — Knowable at the decision moment because March Google Search impressions have already been recorded.
2. `clicks_30d` — Knowable at the decision moment because March Google Search clicks have already been recorded.
3. `avg_position_30d` — Knowable at the decision moment because March average search position has already been recorded.
4. `organic_sessions_30d` — Knowable at the decision moment when GA4 data is available because March organic sessions have already been recorded.
5. `scroll_events_30d` — Knowable at the decision moment when GA4 data is available because March scroll events have already been recorded.

In [12]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

feature_cols = [
    "impressions_30d",
    "clicks_30d",
    "avg_position_30d",
    "organic_sessions_30d",
    "scroll_events_30d"
]

X = model_df[feature_cols]
y = model_df["decline_label"]
groups = model_df["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(splitter.split(X, y, groups=groups))

imputer = SimpleImputer(strategy="median")
X_train = imputer.fit_transform(X.iloc[train_idx])
X_test = imputer.transform(X.iloc[test_idx])

model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(X_train, y.iloc[train_idx])

honest_pred = model.predict(X_test)
honest_score = accuracy_score(y.iloc[test_idx], honest_pred)

# Deliberate leakage: the label itself is added as a feature
X_leak = model_df[feature_cols].copy()
X_leak["leak_feature"] = y

X_leak_train = imputer.fit_transform(X_leak.iloc[train_idx])
X_leak_test = imputer.transform(X_leak.iloc[test_idx])

leak_model = DecisionTreeClassifier(max_depth=4, random_state=42)
leak_model.fit(X_leak_train, y.iloc[train_idx])

leak_pred = leak_model.predict(X_leak_test)
leak_score = accuracy_score(y.iloc[test_idx], leak_pred)

print("Honest accuracy:", round(honest_score, 4))
print("Leaky accuracy:", round(leak_score, 4))

Honest accuracy: 0.8459
Leaky accuracy: 1.0


### Leakage lesson

The leaky score becomes artificially close to perfect because `leak_feature` directly contains the label. That information would not exist at the decision moment. I removed the leaky feature and kept the honest score as the valid result.

In [6]:
q = f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM {REL}
"""

con.sql(q).df()

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


In [7]:
q = f"""
SELECT
    COUNT(*) AS rows_with_gsc_data
FROM {REL}
WHERE gsc_data_available IS TRUE
"""

con.sql(q).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows_with_gsc_data
0,3611061


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Limitation

A limitation of this slice is that data availability differs across clients. Some rows do not have usable GA4 data, so the GA4-based features are not available for every content item.

In [14]:
print("Missing values in the five features:")
print(final_features[feature_cols].isna().sum())

Missing values in the five features:
impressions_30d             0
clicks_30d                  0
avg_position_30d          759
organic_sessions_30d    44696
scroll_events_30d       44696
dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.